# LeetCode 771 — Jewels and Stones bằng Python

**Link đề:** https://leetcode.com/problems/jewels-and-stones/
**Độ khó:** Easy · **Chủ đề:** Hash Table (bảng băm), String

```python
class Solution:
    def numJewelsInStones(self, jewels: str, stones: str) -> int:
        ...
```

## Mục tiêu học
1. Duyệt chuỗi từng ký tự, hiểu **phân biệt hoa/thường** (case sensitive).
2. Thấy chi phí của toán tử `in` trên **chuỗi** (`O(len)`) so với trên **set** (`O(1)` trung bình).
3. Viết 4 lời giải: hai vòng lặp → `in` chuỗi → `set` (chính) → `Counter`; thêm mẹo **bitmask 52 chữ cái**.
4. Trace từng bước, chứng minh đúng, phân tích Time/Space.
5. Nhận dạng pattern "đếm phần tử thuộc một tập" và giải các biến thể.

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài — đếm số viên đá là ngọc

- Chuỗi `jewels`: mỗi ký tự là một **loại** đá được coi là ngọc (các ký tự **khác nhau**).
- Chuỗi `stones`: mỗi ký tự là **một viên đá** bạn đang có.

Hỏi: có bao nhiêu viên trong `stones` là ngọc? Chữ **hoa và thường khác nhau**: `"a"` khác `"A"`.

| jewels | stones | Output | Giải thích |
|---|---|:---:|---|
| `"aA"` | `"aAAbbbb"` | `3` | `a`, `A`, `A` là ngọc; 4 viên `b` thì không |
| `"z"` | `"ZZ"` | `0` | `Z` hoa khác `z` thường |

**Constraints:**
- `1 <= jewels.length, stones.length <= 50`
- Chỉ gồm chữ cái tiếng Anh.
- Các ký tự trong `jewels` đôi một khác nhau.

In [1]:
jewels, stones = "aA", "aAAbbbb"
for i, ch in enumerate(stones):
    print(i, ch, "-> là ngọc" if ch in jewels else "-> không")

0 a -> là ngọc
1 A -> là ngọc
2 A -> là ngọc
3 b -> không
4 b -> không
5 b -> không
6 b -> không


## 2. Nền tảng Python: chuỗi, duyệt ký tự và so sánh hoa/thường

- Chuỗi là dãy ký tự; `for ch in s` lấy lần lượt từng ký tự.
- `"a" == "A"` là `False`: mỗi ký tự có mã Unicode riêng (`ord("a") = 97`, `ord("A") = 65`).
- `ch in s` (với `s` là chuỗi) kiểm tra `ch` có xuất hiện trong `s` không — Python phải **duyệt** `s` từ đầu.

In [2]:
print("a" == "A", ord("a"), ord("A"))
print("A" in "aA", "z" in "aA", "Z" in "z")
for ch in "aAb":
    print(repr(ch))

False 97 65
True False False
'a'
'A'
'b'


### 2.1. Biểu thức điều kiện một dòng `X if cond else Y`

Cell ở mục 1 dùng `"-> là ngọc" if ch in jewels else "-> không"`: nếu điều kiện đúng lấy vế trái, sai lấy vế phải.

In [3]:
for n in [3, 8]:
    print(n, "lớn" if n > 5 else "nhỏ")

3 nhỏ
8 lớn


## 3. Cách 1 — Hai vòng lặp lồng nhau (brute force)

Với mỗi viên đá `s`, duyệt từng loại ngọc `j`; nếu `s == j` thì đếm và dừng (`break`) vì các loại ngọc không trùng nhau.

In [4]:
def num_jewels_bruteforce(jewels: str, stones: str) -> int:
    count = 0
    for s in stones:
        for j in jewels:
            if s == j:
                count += 1
                break            # đã khớp, không cần xét loại ngọc khác
    return count

print(num_jewels_bruteforce("aA", "aAAbbbb"))   # 3
print(num_jewels_bruteforce("z", "ZZ"))         # 0

3
0


### `break` là gì?

`break` thoát ngay khỏi **vòng lặp gần nhất** (vòng trong), quay về vòng ngoài để xét viên đá kế tiếp.

In [5]:
for x in [1, 2, 3, 4]:
    if x == 3:
        print("gặp 3, break")
        break
    print("x =", x)

x = 1
x = 2
gặp 3, break


**Độ phức tạp cách 1:** gọi `J = len(jewels)`, `S = len(stones)`. Mỗi viên đá có thể so với cả `J` loại → `O(J·S)` thời gian, `O(1)` bộ nhớ.

## 4. Cách 2 — Dùng `in` trên chuỗi

Vòng lặp trong của cách 1 chính là việc toán tử `ch in jewels` làm sẵn. Code gọn hơn nhưng **độ phức tạp giống hệt** `O(J·S)`.

In [6]:
def num_jewels_in_string(jewels: str, stones: str) -> int:
    return sum(1 for s in stones if s in jewels)

print(num_jewels_in_string("aA", "aAAbbbb"), num_jewels_in_string("z", "ZZ"))

3 0


### `sum(1 for ... if ...)` — đếm bằng generator

`(1 for s in stones if s in jewels)` sinh số `1` cho mỗi viên thỏa; `sum` cộng lại = số viên thỏa. Viết tương đương: `sum(s in jewels for s in stones)` vì `True` = 1, `False` = 0.

In [7]:
print(sum(1 for s in "aAAbbbb" if s in "aA"))
print(sum(s in "aA" for s in "aAAbbbb"))
print(True + True + False)

3
3
2


## 5. Nút thắt và insight — biến `jewels` thành `set`

Nút thắt: mỗi lần hỏi `s in jewels` ta lại quét lại cả chuỗi `jewels`. Nếu `jewels` có 52 chữ và `stones` rất dài, chi phí là `52 · S`.

Insight: `jewels` **không thay đổi** trong suốt vòng lặp → chuẩn bị một lần thành **set** (bảng băm). Tra `in` trên set trung bình `O(1)`.

```text
set("aA")  ->  {'a', 'A'}
```

In [8]:
jewel_set = set("aA")
print(jewel_set, type(jewel_set))
print("a" in jewel_set, "b" in jewel_set)
print(set("hello"))       # set bỏ trùng lặp, không giữ thứ tự

{'a', 'A'} <class 'set'>
True False
{'h', 'l', 'e', 'o'}


### 5.1. Vì sao `in` trên set nhanh? (ý tưởng bảng băm)

Set tính `hash(ch)` — một số nguyên "đại diện" cho phần tử — rồi dùng nó để nhảy thẳng tới **ô chứa** tương ứng trong bảng nội bộ, thay vì so lần lượt từng phần tử. Vì vậy thời gian tra **không phụ thuộc** số phần tử (trung bình). Trường hợp xấu (nhiều phần tử trùng ô) hiếm gặp, nên ta nói "`O(1)` trung bình".

So sánh trực quan: tìm tên trong danh bạ **chưa sắp xếp** (phải đọc từng dòng) với tra từ điển đã có **mục lục theo chữ cái** (mở thẳng trang).

In [9]:
print(hash("a") == hash("a"), hash("a") == hash("A"))   # cùng ký tự -> cùng hash
big_list = list(range(1_000_000))
big_set = set(big_list)
import time
t = time.perf_counter(); _ = 999_999 in big_list; t_list = time.perf_counter() - t
t = time.perf_counter(); _ = 999_999 in big_set;  t_set = time.perf_counter() - t
print(f"in list: {t_list*1e3:.3f} ms | in set: {t_set*1e3:.4f} ms")

True False
in list: 7.060 ms | in set: 0.0359 ms


## 6. Lời giải chính — `set` + một vòng duyệt

Giảng từng đoạn:
```python
jewel_set = set(jewels)      # (1) chuẩn bị tập ngọc: O(J)
count = 0                    # (2) bộ đếm
for s in stones:             # (3) duyệt từng viên đá: S lần
    if s in jewel_set:       # (4) tra O(1) trung bình
        count += 1
return count
```

In [10]:
def num_jewels(jewels: str, stones: str) -> int:
    jewel_set = set(jewels)
    count = 0
    for s in stones:
        if s in jewel_set:
            count += 1
    return count

print(num_jewels("aA", "aAAbbbb"), num_jewels("z", "ZZ"))

3 0


## 7. TRACE từng bước

`jewels = "aA"`, `stones = "aAAbbbb"`, `jewel_set = {'a','A'}`:

| i | s | s in jewel_set? | count trước → sau |
|---|---|:---:|---|
| 0 | a | ✔ | 0 → 1 |
| 1 | A | ✔ | 1 → 2 |
| 2 | A | ✔ | 2 → 3 |
| 3 | b | ✘ | 3 → 3 |
| 4 | b | ✘ | 3 → 3 |
| 5 | b | ✘ | 3 → 3 |
| 6 | b | ✘ | 3 → 3 |

In [11]:
def trace_num_jewels(jewels: str, stones: str) -> int:
    jewel_set = set(jewels)
    print("jewel_set =", sorted(jewel_set))
    count = 0
    for i, s in enumerate(stones):
        before = count
        hit = s in jewel_set
        if hit:
            count += 1
        print(f"i={i} s={s!r} trong tập? {str(hit):<5} count {before} -> {count}")
    print("Kết quả:", count)
    return count

_ = trace_num_jewels("aA", "aAAbbbb")
print()
_ = trace_num_jewels("z", "ZZ")

jewel_set = ['A', 'a']
i=0 s='a' trong tập? True  count 0 -> 1
i=1 s='A' trong tập? True  count 1 -> 2
i=2 s='A' trong tập? True  count 2 -> 3
i=3 s='b' trong tập? False count 3 -> 3
i=4 s='b' trong tập? False count 3 -> 3
i=5 s='b' trong tập? False count 3 -> 3
i=6 s='b' trong tập? False count 3 -> 3
Kết quả: 3

jewel_set = ['z']
i=0 s='Z' trong tập? False count 0 -> 0
i=1 s='Z' trong tập? False count 0 -> 0
Kết quả: 0


`{s!r}` trong f-string in **repr** của giá trị — với chuỗi sẽ có dấu nháy `'a'`, giúp phân biệt rõ ký tự (nhất là dấu cách).

## 8. Code nộp LeetCode — lời giải chính

In [12]:
class Solution:
    def numJewelsInStones(self, jewels: str, stones: str) -> int:
        jewel_set = set(jewels)
        return sum(s in jewel_set for s in stones)

sol = Solution()
assert sol.numJewelsInStones("aA", "aAAbbbb") == 3
assert sol.numJewelsInStones("z", "ZZ") == 0
print("Solution OK")

Solution OK


## 9. Cách 4 — Đếm loại đá bằng `Counter`, rồi cộng theo ngọc

Nhìn từ phía ngược lại: đếm mỗi loại đá xuất hiện bao nhiêu lần (`Counter(stones)`), rồi cộng số lượng của các loại là ngọc.

`Counter` (module chuẩn `collections`) là dict đặc biệt: key là phần tử, value là số lần xuất hiện; key không có trả về `0` thay vì lỗi.

In [13]:
from collections import Counter
cnt = Counter("aAAbbbb")
print(cnt)
print("cnt['A'] =", cnt["A"], "| cnt['z'] =", cnt["z"])   # không có -> 0, không KeyError

def num_jewels_counter(jewels: str, stones: str) -> int:
    cnt = Counter(stones)
    return sum(cnt[j] for j in jewels)

print(num_jewels_counter("aA", "aAAbbbb"), num_jewels_counter("z", "ZZ"))

Counter({'b': 4, 'A': 2, 'a': 1})
cnt['A'] = 2 | cnt['z'] = 0
3 0


In [14]:
class Solution:
    def numJewelsInStones(self, jewels: str, stones: str) -> int:
        cnt = Counter(stones)
        return sum(cnt[j] for j in jewels)

print(Solution().numJewelsInStones("aA", "aAAbbbb"))

3


Phiên bản "một dòng" hay gặp: `sum(map(stones.count, jewels))` — với mỗi loại ngọc, đếm số lần trong `stones`. Ngắn nhưng là `O(J·S)` (mỗi `count` quét lại cả `stones`).

`map(f, iterable)` áp dụng hàm `f` cho từng phần tử.

In [15]:
print(list(map(len, ["ab", "c", "def"])))
print(list(map("aAAbbbb".count, "aA")))
print(sum(map("aAAbbbb".count, "aA")))

[2, 1, 3]
[1, 2]
3


## 10. Cách 5 (mẹo) — Bitmask 52 chữ cái

Có đúng 52 chữ cái (26 thường + 26 hoa). Gán mỗi chữ một **vị trí bit**; tập ngọc là một số nguyên `mask` có bit bật tại vị trí của các ngọc. Tra "là ngọc?" = `(mask >> pos) & 1`.

Dùng `ord(ch) - ord('A')` cho vị trí: `'A'..'Z'` → 0..25, `'a'..'z'` → 32..57 (giữa có 6 ký tự khác, không sao — số nguyên Python đủ lớn).

In [16]:
def num_jewels_bitmask(jewels: str, stones: str) -> int:
    mask = 0
    for j in jewels:
        mask |= 1 << (ord(j) - ord("A"))
    return sum((mask >> (ord(s) - ord("A"))) & 1 for s in stones)

print("vị trí 'A' =", ord("A") - 65, "| 'Z' =", ord("Z") - 65, "| 'a' =", ord("a") - 65, "| 'z' =", ord("z") - 65)
print(num_jewels_bitmask("aA", "aAAbbbb"), num_jewels_bitmask("z", "ZZ"))

vị trí 'A' = 0 | 'Z' = 25 | 'a' = 32 | 'z' = 57
3 0


## 11. Vì sao đúng?

**Invariant:** sau khi xét `k` viên đầu của `stones`, `count` = số viên trong `stones[:k]` là ngọc.
- `k = 0`: `count = 0` đúng.
- Viên thứ `k`: cộng 1 **khi và chỉ khi** `s in jewel_set`, mà `jewel_set` chứa đúng các ký tự của `jewels` (set so sánh **chính xác** ký tự, nên phân biệt hoa/thường tự động).
- Hết vòng: `count` là đáp án.

Cách `Counter` đúng vì mỗi viên đá thuộc **đúng một** loại; các loại ngọc khác nhau nên không viên nào bị cộng hai lần: `Σ_{j ∈ jewels} cnt[j]` = số viên là ngọc.

## 12. Độ phức tạp và so sánh

`J = len(jewels)` (≤ 50), `S = len(stones)` (≤ 50).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Hai vòng lặp | `O(J·S)` | `O(1)` | Dễ nghĩ |
| 2. `in` trên chuỗi | `O(J·S)` | `O(1)` | Gọn, nhưng vẫn quét |
| 3. `set` (chính) | `O(J + S)` trung bình | `O(J)` | **Khuyến nghị** |
| 4. `Counter(stones)` | `O(J + S)` | `O(S)` (≤ 52 key) | Nhìn từ phía đá |
| 4b. `map(stones.count, jewels)` | `O(J·S)` | `O(1)` | Một dòng |
| 5. Bitmask | `O(J + S)` | `O(1)` | Mẹo, chỉ khi bảng chữ nhỏ |

Với constraints ≤ 50, cách nào cũng nhanh; nhưng cách `set` là thói quen đúng khi dữ liệu lớn.

In [17]:
import time, random, string
random.seed(771)
J = "".join(random.sample(string.ascii_letters, 52))
S = "".join(random.choice(string.ascii_letters) for _ in range(200_000))
for f in [num_jewels_bruteforce, num_jewels_in_string, num_jewels, num_jewels_counter, num_jewels_bitmask]:
    t = time.perf_counter()
    r = f(J[:26], S)
    print(f"{f.__name__:<24} -> {r}  ({(time.perf_counter() - t) * 1000:.1f} ms)")

num_jewels_bruteforce    -> 100166  (118.4 ms)
num_jewels_in_string     -> 100166  (8.1 ms)
num_jewels               -> 100166  (7.1 ms)
num_jewels_counter       -> 100166  (6.4 ms)
num_jewels_bitmask       -> 100166  (25.4 ms)


**Đọc kết quả đo cẩn thận:** cách 1 (hai vòng lặp viết bằng Python) chậm rõ rệt. Cách 2 (`in` trên chuỗi) trông nhanh ngang cách `set` vì vòng quét `jewels` chạy bằng C và `jewels` chỉ có 26 ký tự — hằng số nhỏ. Về mặt **tiệm cận** nó vẫn là `O(J·S)`: nếu tập "ngọc" có hàng nghìn phần tử (ví dụ tập từ khóa), set sẽ thắng xa. Bài học: Big-O mô tả xu hướng khi dữ liệu lớn, còn thời gian thực đo trên dữ liệu nhỏ có thể bị hằng số chi phối.

`string.ascii_letters` là hằng số chuỗi `'abc...zABC...Z'` (52 chữ); `random.sample(seq, k)` lấy `k` phần tử **không lặp**; `random.choice(seq)` lấy 1 phần tử ngẫu nhiên.

## 13. Bộ test

1. Ví dụ đề.
2. Biên: một ký tự mỗi chuỗi, khớp / không khớp.
3. Phân biệt hoa/thường (`"a"` vs `"A"`).
4. Tất cả đều là ngọc / không viên nào là ngọc.
5. Đối chiếu các cách trên dữ liệu ngẫu nhiên.

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

check("đề 1", num_jewels("aA", "aAAbbbb"), 3)
check("đề 2", num_jewels("z", "ZZ"), 0)
check("1 ký tự khớp", num_jewels("x", "x"), 1)
check("1 ký tự không khớp", num_jewels("x", "y"), 0)
check("hoa/thường", num_jewels("a", "AaAaA"), 2)
check("toàn ngọc", num_jewels("abc", "cabbac"), 6)
check("không ngọc", num_jewels("XYZ", "xyzxyz"), 0)

PASS đề 1: 3
PASS đề 2: 0
PASS 1 ký tự khớp: 1
PASS 1 ký tự không khớp: 0
PASS hoa/thường: 2
PASS toàn ngọc: 6
PASS không ngọc: 0


In [19]:
methods = [num_jewels_bruteforce, num_jewels_in_string, num_jewels, num_jewels_counter, num_jewels_bitmask,
           lambda j, s: Solution().numJewelsInStones(j, s)]
for _ in range(500):
    jw = "".join(random.sample(string.ascii_letters, random.randint(1, 10)))
    st = "".join(random.choice(string.ascii_letters[:8] + string.ascii_letters[26:34]) for _ in range(random.randint(1, 50)))
    results = {f(jw, st) for f in methods}
    assert len(results) == 1, (jw, st, results)
print("500 bộ ngẫu nhiên: 6 cách khớp nhau")

500 bộ ngẫu nhiên: 6 cách khớp nhau


`lambda j, s: ...` là **hàm vô danh** viết một dòng; ở đây bọc `Solution().numJewelsInStones` để có cùng dạng gọi `f(jw, st)` như các hàm khác.

## 14. Lỗi người mới hay gặp

### Lỗi 1: Đổi về chữ thường "cho chắc"
```python
jewels, stones = jewels.lower(), stones.lower()   # SAI: đề phân biệt hoa/thường
```

### Lỗi 2: Đếm ngược chiều (đếm ngọc có trong đá)
```python
return sum(j in stones for j in jewels)   # SAI: đếm số LOẠI ngọc có mặt, không phải số VIÊN
```

### Lỗi 3: Tạo set **bên trong** vòng lặp
```python
for s in stones:
    if s in set(jewels): ...   # đúng kết quả nhưng tạo lại set mỗi lần -> O(J·S)
```

In [20]:
def wrong_lower(jewels, stones):
    return num_jewels(jewels.lower(), stones.lower())

def wrong_direction(jewels, stones):
    return sum(j in stones for j in jewels)

print("z / ZZ  : wrong_lower =", wrong_lower("z", "ZZ"), "| đúng =", num_jewels("z", "ZZ"))
print("aA / aAAbbbb : wrong_direction =", wrong_direction("aA", "aAAbbbb"), "| đúng =", num_jewels("aA", "aAAbbbb"))

z / ZZ  : wrong_lower = 2 | đúng = 0
aA / aAAbbbb : wrong_direction = 2 | đúng = 3


## 15. Mẹo nhận dạng pattern

- "Đếm/kiểm tra phần tử của A **có thuộc** B" + B cố định ⇒ chuyển B thành **set** trước vòng lặp.
- "Đếm số lần xuất hiện" ⇒ `Counter` / dict đếm.
- Bảng chữ cái nhỏ, cố định (26/52 ký tự) ⇒ có thể dùng **mảng 128 phần tử** hoặc **bitmask** thay hash.
- Đề nhấn mạnh "case sensitive" ⇒ **không** gọi `.lower()`.

### Biến thể

1. Trả về **danh sách viên ngọc** (giữ thứ tự) thay vì số lượng.
2. Đếm số viên **không** phải ngọc.
3. Đếm ngọc **không phân biệt hoa/thường**.
4. LeetCode 1832 (Pangram), 383 (Ransom Note), 2085... — cùng tư duy set/Counter.

In [21]:
# Biến thể 1 và 3
def jewels_list(jewels, stones):
    js = set(jewels)
    return [s for s in stones if s in js]

def num_jewels_ignore_case(jewels, stones):
    js = set(jewels.lower())
    return sum(s.lower() in js for s in stones)

print(jewels_list("aA", "aAAbbbb"))
print(num_jewels_ignore_case("z", "ZZ"))

['a', 'A', 'A']
2


## 16. Bài tập tự luyện (TODO — đáp án ở cuối)

**Bài 1 (đoán output):** Không chạy code, cho biết `num_jewels("bB", "aBcbBB")` trả về bao nhiêu. Gán vào `answer_ex1`.

**Bài 2 (điền chỗ trống):** Hoàn thiện `num_jewels_exercise` dùng set.

**Bài 3 (biến thể):** `count_non_jewels(jewels, stones)` đếm số viên **không** phải ngọc.

**Bài 4 (Counter):** `most_common_jewel(jewels, stones)` trả về loại ngọc xuất hiện nhiều nhất trong `stones` (nếu hòa, lấy loại đứng trước trong `jewels`; nếu không có ngọc nào, trả `""`).

**Bài 5 (mini challenge):** `num_jewels_array(jewels, stones)` không dùng `set`/`dict`/`Counter`: dùng một **list 128 phần tử** `is_jewel[ord(ch)]`.

In [22]:
answer_ex1 = None   # TODO

def num_jewels_exercise(jewels: str, stones: str) -> int:
    # TODO 1: tạo set từ jewels
    # TODO 2: duyệt stones, đếm viên thuộc set
    pass

def count_non_jewels(jewels: str, stones: str) -> int:
    # TODO
    pass

# assert num_jewels_exercise("aA", "aAAbbbb") == 3
# assert count_non_jewels("aA", "aAAbbbb") == 4

In [23]:
def most_common_jewel(jewels: str, stones: str) -> str:
    # TODO
    pass

def num_jewels_array(jewels: str, stones: str) -> int:
    # TODO: is_jewel = [False] * 128
    pass

# assert most_common_jewel("aA", "aAAbbbb") == "A"
# assert num_jewels_array("aA", "aAAbbbb") == 3

### Gợi ý 3 mức

**Bài 1** — (1) Ngọc là `b` và `B`. (2) Duyệt `a B c b B B`. (3) Đếm: `B`, `b`, `B`, `B`.

**Bài 2** — (1) Set trước vòng lặp. (2) `js = set(jewels)`. (3) `sum(s in js for s in stones)`.

**Bài 3** — (1) Phần bù. (2) `len(stones) - số_ngọc`. (3) Hoặc `sum(s not in js for s in stones)`.

**Bài 4** — (1) Đếm bằng `Counter(stones)`. (2) Duyệt `jewels` theo thứ tự, giữ loại có số đếm lớn nhất (dùng `>` để giữ loại đứng trước khi hòa). (3) Nếu số đếm lớn nhất là 0 → `""`.

**Bài 5** — (1) Mỗi ký tự ASCII có `ord < 128`. (2) Đánh dấu `is_jewel[ord(j)] = True`. (3) `sum(is_jewel[ord(s)] for s in stones)`.

---

## 17. Đáp án tham khảo

In [24]:
answer_ex1_solution = 4

def num_jewels_exercise_solution(jewels, stones):
    js = set(jewels)
    return sum(s in js for s in stones)

def count_non_jewels_solution(jewels, stones):
    js = set(jewels)
    return sum(s not in js for s in stones)

def most_common_jewel_solution(jewels, stones):
    cnt = Counter(stones)
    best, best_count = "", 0
    for j in jewels:
        if cnt[j] > best_count:
            best, best_count = j, cnt[j]
    return best

def num_jewels_array_solution(jewels, stones):
    is_jewel = [False] * 128
    for j in jewels:
        is_jewel[ord(j)] = True
    return sum(is_jewel[ord(s)] for s in stones)

assert answer_ex1_solution == num_jewels("bB", "aBcbBB")
assert num_jewels_exercise_solution("aA", "aAAbbbb") == 3
assert count_non_jewels_solution("aA", "aAAbbbb") == 4
assert most_common_jewel_solution("aA", "aAAbbbb") == "A"
assert most_common_jewel_solution("ab", "abab") == "a"      # hòa -> loại đứng trước
assert most_common_jewel_solution("z", "ZZ") == ""
assert num_jewels_array_solution("aA", "aAAbbbb") == 3
assert num_jewels_array_solution("z", "ZZ") == 0
print("Đáp án PASS toàn bộ test!")

Đáp án PASS toàn bộ test!


## 18. Template nhớ nhanh và checklist submit

```python
class Solution:
    def numJewelsInStones(self, jewels: str, stones: str) -> int:
        jewel_set = set(jewels)                      # chuẩn bị 1 lần
        return sum(s in jewel_set for s in stones)   # tra O(1) mỗi viên
```

Pattern: **"tập cố định + nhiều truy vấn thuộc/không thuộc" ⇒ `set`**.

**Checklist trước khi submit:**
- [ ] Không đổi hoa/thường (đề case sensitive)?
- [ ] Đếm theo **viên đá** (duyệt `stones`), không theo loại ngọc?
- [ ] Set được tạo **một lần**, ngoài vòng lặp?
- [ ] Đã thử `("aA","aAAbbbb") → 3`, `("z","ZZ") → 0`?